In [1]:
import pandas as pd
import numpy as np
import os


os.chdir("C:/Users/cuent/En-Peu/notebooks/")
print(os.getcwd())

C:\Users\cuent\En-Peu\notebooks


In [2]:
# Cambiar al directorio padre
os.chdir("..")
parent_directory = os.getcwd()


# os.path.join() maneja correctamente las barras de ruta para diferentes SO (Windows, Linux, macOS)
file_path = os.path.join(parent_directory, 'data', 'data_con_embeddings.csv')


# Cargar el archivo CSV
df = pd.read_csv(file_path)


# Verificar la estructura del DataFrame (equivalente a str(data) en R)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64345 entries, 0 to 64344
Data columns (total 85 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   POINT_X  64345 non-null  float64
 1   POINT_Y  64345 non-null  float64
 2   PLA      64345 non-null  float64
 3   A        64345 non-null  float64
 4   AC       64345 non-null  float64
 5   CIB      64345 non-null  float64
 6   CICCB    64345 non-null  float64
 7   DBB      64345 non-null  float64
 8   DOB      64345 non-null  float64
 9   P        64345 non-null  float64
 10  CUS_1    64345 non-null  float64
 11  CUS_2    64345 non-null  float64
 12  CUS_3    64345 non-null  float64
 13  CUS_4    64345 non-null  float64
 14  CUS_5    64345 non-null  float64
 15  OC_1     64345 non-null  float64
 16  OC_2     64345 non-null  float64
 17  OC_3     64345 non-null  float64
 18  OC_4     64345 non-null  float64
 19  lon      64345 non-null  float64
 20  lat      64345 non-null  float64
 21  emb_0    643

In [4]:
# =========================
# CÓDIGO COMPLETO INTEGRADO - HRM Mejorado
# =========================

# 1. INSTALACIONES Y IMPORTS (tu código original)
!pip -q install scikit-learn pandas numpy matplotlib tqdm
!pip install sympy>=1.13.3


import os, math, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset




from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor


print("Shape:", df.shape)
print("Columnas:", df.columns.tolist()[:40])

# Validaciones
assert "PLA" in df.columns, "Falta la columna objetivo PLA"

# Definir columnas (tu código original)
emb_cols = [c for c in df.columns if c.lower().startswith("emb_")]
geo_cols = ["A","CIB","CICCB","DOB","DBB","P"] 
#+ \
#           [c for c in df.columns if c.startswith("OC_") or c.startswith("CUS_")]
coord_cols = ["POINT_X","POINT_Y"]

print(f"Embeddings detectados: {len(emb_cols)}")
print(f"Geo cols detectadas: {len(geo_cols)}")

# 4. PREPARAR FEATURES (tu código original)
def prepare_features(df, use_emb=True, use_geo=True, use_coords=False):
    feats = []
    if use_geo:
        feats += [c for c in geo_cols if c in df.columns]
    if use_emb:
        feats += emb_cols
    if use_coords:
        feats += [c for c in coord_cols if c in df.columns]
    print(f"Usando {len(feats)} features ({feats[:10]} ...)")
    X = df[feats].astype(np.float32).fillna(0.0).values
    y = df["PLA"].astype(np.float32).values
    return X,y,feats



Shape: (64345, 85)
Columnas: ['POINT_X', 'POINT_Y', 'PLA', 'A', 'AC', 'CIB', 'CICCB', 'DBB', 'DOB', 'P', 'CUS_1', 'CUS_2', 'CUS_3', 'CUS_4', 'CUS_5', 'OC_1', 'OC_2', 'OC_3', 'OC_4', 'lon', 'lat', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18']
Embeddings detectados: 64
Geo cols detectadas: 6


In [5]:
# 5. PREPARAR DATOS CON OPCIONES FLEXIBLES
# =========================
# CONFIGURACIÓN DE FEATURES - CAMBIA AQUÍ LO QUE QUIERAS USAR
# =========================

# CONTROLES PRINCIPALES - Cambia estos valores según tus experimentos:
USE_EMBEDDINGS = False      #  Usar embeddings AlphaEarth
USE_GEO = True             #  Usar variables geoespaciales
USE_COORDS = False         #  Usar coordenadas X,Y

# EXPERIMENTOS RÁPIDOS - Descomenta la línea que quieras probar:
# USE_EMBEDDINGS, USE_GEO, USE_COORDS = True,  False, False  # Solo embeddings
# USE_EMBEDDINGS, USE_GEO, USE_COORDS = False, True,  False  # Solo geo (tu config original)
USE_EMBEDDINGS, USE_GEO, USE_COORDS = True,  True,  False  # Embeddings + Geo (recomendado)
# USE_EMBEDDINGS, USE_GEO, USE_COORDS = True,  True,  True   # Todo (máxima información)

print("\n" + "="*60)
print("CONFIGURACIÓN DE FEATURES:")
print(f"Embeddings AlphaEarth: {'SÍ' if USE_EMBEDDINGS else 'NO'}")
print(f"Variables geoespaciales: {'SÍ' if USE_GEO else 'NO'}")
print(f"Coordenadas X,Y: {'SÍ' if USE_COORDS else 'NO'}")
print("="*60)

# Preparar features según configuración
X, y, feats = prepare_features(df,
                              use_emb=USE_EMBEDDINGS,
                              use_geo=USE_GEO,
                              use_coords=USE_COORDS)

# Splits
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, random_state=42)
Xva, Xte, yva, yte   = train_test_split(Xtmp, ytmp, test_size=0.50, random_state=42)

print(f"\n DIMENSIONES:")
print(f"Train: {Xtr.shape}, Val: {Xva.shape}, Test: {Xte.shape}")
print(f"Features seleccionadas: {len(feats)}")


# =========================
# 6. AQUÍ EMPIEZA EL CÓDIGO MEJORADO
# =========================

class ImprovedHRMRegressor(nn.Module):
    """HRM mejorado para alcanzar R² > 0.97"""
    def __init__(
        self,
        input_dim,
        hidden_high=256,
        hidden_low=512,
        plan_dim=64,
        K=16,
        p_dropout=0.15,
        use_deep_supervision=True,
        use_halting=True,
        use_residual=True,
        use_attention=True
    ):
        super().__init__()
        self.K = K
        self.use_deep_supervision = use_deep_supervision
        self.use_halting = use_halting
        self.use_residual = use_residual
        self.use_attention = use_attention

        # Proyección con bottleneck (SIN BatchNorm - datos ya normalizados)
        self.inp_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_low * 2),
            nn.GELU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_low * 2, hidden_low)
        )
        self.inp_ln = nn.LayerNorm(hidden_low)

        # Alto nivel mejorado
        self.gru_high = nn.GRU(hidden_low, hidden_high, batch_first=True, dropout=p_dropout if K > 1 else 0)
        self.high_ln = nn.LayerNorm(hidden_high)

        # Plan generation con más capacidad
        self.to_plan = nn.Sequential(
            nn.Linear(hidden_high, plan_dim * 2),
            nn.GELU(),
            nn.Dropout(p_dropout),
            nn.Linear(plan_dim * 2, plan_dim)
        )

        # Bajo nivel mejorado
        self.gru_low = nn.GRU(hidden_low + plan_dim, hidden_low, batch_first=True, dropout=p_dropout if K > 1 else 0)
        self.low_ln = nn.LayerNorm(hidden_low)

        # Attention mechanism
        if self.use_attention:
            self.attention = nn.MultiheadAttention(
                embed_dim=hidden_low,
                num_heads=8,
                dropout=p_dropout,
                batch_first=True
            )
            self.att_ln = nn.LayerNorm(hidden_low)

        # Residual projections
        if self.use_residual:
            self.residual_proj = nn.Linear(hidden_low, hidden_low)

        self.dropout = nn.Dropout(p_dropout)

        # Improved prediction head
        self.head = nn.Sequential(
            nn.Linear(hidden_low, hidden_low // 2),
            nn.GELU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_low // 2, 1)
        )

        # Halting con mejoras
        if self.use_halting:
            self.halt_head = nn.Sequential(
                nn.Linear(hidden_low, hidden_low // 4),
                nn.GELU(),
                nn.Linear(hidden_low // 4, 1)
            )

        # Mejor inicialización
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.GRU):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_normal_(param)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0)

    def forward(self, x):
        B = x.size(0)

        # Sin normalización adicional - datos ya están normalizados con MinMaxScaler
        h_low = self.inp_proj(x)
        h_low = self.inp_ln(h_low).unsqueeze(1)

        # Estado inicial del planificador
        h_high = torch.zeros(1, B, self.gru_high.hidden_size, device=x.device)

        preds_all = []
        gates_all, rema_all = [], []
        remainder = torch.ones(B, device=x.device)

        low_states = []  # Para attention

        for t in range(self.K):
            # Planificador
            high_out, h_high = self.gru_high(h_low, h_high)
            high_out = self.high_ln(high_out)

            # Plan más sofisticado
            plan = torch.tanh(self.to_plan(high_out.squeeze(1))).unsqueeze(1)

            # Ejecutor
            low_in = torch.cat([h_low, plan], dim=-1)
            low_in = self.dropout(low_in)
            low_out, _ = self.gru_low(low_in)
            low_out = self.low_ln(low_out)

            # Residual connection
            if self.use_residual and t > 0:
                low_out = low_out + self.residual_proj(h_low)

            # Store for attention
            low_states.append(low_out)

            # Self-attention entre estados pasados
            if self.use_attention and len(low_states) > 1:
                states_seq = torch.cat(low_states, dim=1)  # [B, t+1, H]
                att_out, _ = self.attention(states_seq, states_seq, states_seq)
                low_out = self.att_ln(low_out + att_out[:, -1:, :])

            # Predicción
            pred_t = self.head(low_out).reshape(B)
            preds_all.append(pred_t)

            # Halting mejorado
            if self.use_halting:
                halt_logit = self.halt_head(low_out).reshape(B)
                # Temperatura adaptiva
                temp = max(0.1, 1.0 - 0.8 * t / self.K)
                gate_t = torch.sigmoid(halt_logit / temp)
                gates_all.append(gate_t)
                rema_all.append(remainder.clone())
                remainder = (remainder * (1.0 - gate_t)).clamp(0.0, 1.0)

            h_low = low_out

        # Halting final
        if self.use_halting:
            gates = torch.stack(gates_all, dim=0)
            rema = torch.stack(rema_all, dim=0)
            weights = rema * gates
            wsum = weights.sum(dim=0)
            remainder_final = (1.0 - wsum).clamp(min=0.0)
            weights[-1] = weights[-1] + remainder_final

            preds_stack = torch.stack(preds_all, dim=0)
            out_final = (weights * preds_stack).sum(dim=0)

            halting_reg = torch.mean((wsum - 1.0) ** 2)
        else:
            weights = None
            out_final = preds_all[-1]
            halting_reg = torch.tensor(0.0, device=x.device)

        return out_final, preds_all, weights, halting_reg

    def get_step_states(self, x):
        """Para extraer embeddings"""
        self.eval()
        B = x.size(0)

        with torch.no_grad():
            h_low = self.inp_proj(x)
            h_low = self.inp_ln(h_low).unsqueeze(1)

            h_high = torch.zeros(1, B, self.gru_high.hidden_size, device=x.device)

            low_list, high_list, plan_list = [], [], []
            gates_all, rema_all = [], []
            remainder = torch.ones(B, device=x.device)
            preds = []
            low_states = []

            for t in range(self.K):
                high_out, h_high = self.gru_high(h_low, h_high)
                high_out = self.high_ln(high_out)

                plan = torch.tanh(self.to_plan(high_out.squeeze(1))).unsqueeze(1)
                low_in = torch.cat([h_low, plan], dim=-1)
                low_in = self.dropout(low_in)
                low_out, _ = self.gru_low(low_in)
                low_out = self.low_ln(low_out)

                if self.use_residual and t > 0:
                    low_out = low_out + self.residual_proj(h_low)

                low_states.append(low_out)
                if self.use_attention and len(low_states) > 1:
                    states_seq = torch.cat(low_states, dim=1)
                    att_out, _ = self.attention(states_seq, states_seq, states_seq)
                    low_out = self.att_ln(low_out + att_out[:, -1:, :])

                low_list.append(low_out.squeeze(1))
                high_list.append(high_out.squeeze(1))
                plan_list.append(plan.squeeze(1))
                preds.append(self.head(low_out).reshape(B))

                if self.use_halting:
                    halt_logit = self.halt_head(low_out).reshape(B)
                    temp = max(0.1, 1.0 - 0.8 * t / self.K)
                    gate_t = torch.sigmoid(halt_logit / temp)
                    gates_all.append(gate_t)
                    rema_all.append(remainder.clone())
                    remainder = (remainder * (1.0 - gate_t)).clamp(0.0, 1.0)

                h_low = low_out

            low_seq = torch.stack(low_list, dim=0)
            high_seq = torch.stack(high_list, dim=0)
            plan_seq = torch.stack(plan_list, dim=0)

            if self.use_halting:
                gates = torch.stack(gates_all, dim=0)
                rema = torch.stack(rema_all, dim=0)
                weights = rema * gates
                wsum = weights.sum(dim=0)
                remainder_final = (1.0 - wsum).clamp(min=0.0)
                weights[-1] = weights[-1] + remainder_final

                preds = torch.stack(preds, dim=0)
                out_final = (weights * preds).sum(dim=0)
            else:
                weights = None
                out_final = self.head(low_seq[-1]).reshape(B)

        return {
            "low_seq": low_seq,
            "high_seq": high_seq,
            "plan_seq": plan_seq,
            "weights": weights,
            "out_final": out_final
        }


def train_improved_hrm(X, y, X_val=None, y_val=None,
                      epochs=300, batch=512, lr=2e-4,
                      hidden_high=384, hidden_low=768, plan_dim=96,
                      K=16, p_dropout=0.15,
                      deep_supervision=True, use_halting=True,
                      alpha_ds=0.6, beta_halt=0.01, patience=30,
                      warmup_epochs=20):
    """Entrenamiento mejorado - SIN escalado adicional (datos ya normalizados)"""
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Dataset (datos ya normalizados con MinMaxScaler)
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=batch, shuffle=True, num_workers=2)

    if X_val is not None:
        val_dataset = TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32)
        )
        val_loader = DataLoader(val_dataset, batch_size=batch*2, shuffle=False)

    # Modelo mejorado
    model = ImprovedHRMRegressor(
        input_dim=X.shape[1],
        hidden_high=hidden_high,
        hidden_low=hidden_low,
        plan_dim=plan_dim,
        K=K,
        p_dropout=p_dropout,
        use_deep_supervision=deep_supervision,
        use_halting=use_halting
    ).to(device)

    # Optimizer con weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    # Scheduler con warmup
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        else:
            return 0.95 ** ((epoch - warmup_epochs) // 10)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    loss_fn = nn.HuberLoss(delta=0.1)  # Más robusto que MSE

    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []

    print("Iniciando entrenamiento HRM mejorado...")
    for epoch in range(epochs):
        # Training
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            out_final, preds_all, weights, halting_reg = model(xb)

            # Loss principal
            main_loss = loss_fn(out_final, yb)

            # Deep supervision
            ds_loss = 0
            if deep_supervision and len(preds_all) > 1:
                for pred_t in preds_all[:-1]:
                    ds_loss += loss_fn(pred_t, yb)
                ds_loss /= (len(preds_all) - 1)

            # Loss total
            total_loss = main_loss + alpha_ds * ds_loss + beta_halt * halting_reg

            optimizer.zero_grad()
            total_loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            epoch_loss += total_loss.item()

        scheduler.step()
        avg_train_loss = epoch_loss / len(loader)
        train_losses.append(avg_train_loss)

        # Validation
        if X_val is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    out_final, _, _, _ = model(xb)
                    val_loss += loss_fn(out_final, yb).item()

            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_state = model.state_dict().copy()
            else:
                patience_counter += 1

            if epoch % 20 == 0:
                print(f"Epoch {epoch}: Train Loss {avg_train_loss:.6f}, Val Loss {avg_val_loss:.6f}")

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                model.load_state_dict(best_state)
                break

    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses if X_val is not None else [],
        'best_val_loss': best_val_loss
    }


def _pool_seq(seq_tbh, weights_tb=None, mode="weighted_last"):
    """Pooling de secuencias"""
    T, B, H = seq_tbh.shape
    if mode == "weighted" or (mode == "weighted_last" and weights_tb is not None):
        w = weights_tb.unsqueeze(-1)
        out = (seq_tbh * w).sum(dim=0)
        return out
    elif mode == "last" or (mode == "weighted_last" and weights_tb is None):
        return seq_tbh[-1]
    elif mode == "mean":
        return seq_tbh.mean(dim=0)
    else:
        raise ValueError("Modo de pooling no reconocido.")


def hrm_transform(model, X_np, batch=1024, pool_mode="weighted_last", concat_original=False):
    """Extraer embeddings del HRM (sin scaler - datos ya normalizados)"""
    model.eval()
    E_list = []
    device = next(model.parameters()).device

    with torch.no_grad():
        dl = DataLoader(torch.tensor(X_np, device=device, dtype=torch.float32), batch_size=batch, shuffle=False)
        for xb in dl:
            states = model.get_step_states(xb)
            low_seq = states["low_seq"]     # [T,B,Hlow]
            high_seq = states["high_seq"]    # [T,B,Hhigh]
            plan_seq = states["plan_seq"]    # [T,B,Plan]
            weights = states["weights"]      # [T,B] o None

            low_emb = _pool_seq(low_seq, weights, mode=pool_mode)   # [B,Hlow]
            high_emb = _pool_seq(high_seq, weights, mode=pool_mode)   # [B,Hhigh]
            plan_emb = _pool_seq(plan_seq, weights, mode=pool_mode)   # [B,Plan]

            emb = torch.cat([low_emb, high_emb, plan_emb], dim=1).cpu().numpy()   # [B, Hlow+Hhigh+Plan]

            if concat_original:
                emb = np.concatenate([emb, xb.cpu().numpy()], axis=1)

            E_list.append(emb)

    E = np.vstack(E_list)
    return E





CONFIGURACIÓN DE FEATURES:
Embeddings AlphaEarth: SÍ
Variables geoespaciales: SÍ
Coordenadas X,Y: NO
Usando 70 features (['A', 'CIB', 'CICCB', 'DOB', 'DBB', 'P', 'emb_0', 'emb_1', 'emb_2', 'emb_3'] ...)

 DIMENSIONES:
Train: (45041, 70), Val: (9652, 70), Test: (9652, 70)
Features seleccionadas: 70


In [6]:
# =========================
# 7. EJECUTAR PIPELINE MEJORADO CON CONFIGURACIÓN DINÁMICA
# =========================

# Determinar configuración de entrenamiento basada en features
config_name = []
if USE_EMBEDDINGS:
    config_name.append("Embeddings")
if USE_GEO:
    config_name.append("Geo")
if USE_COORDS:
    config_name.append("Coords")

config_str = " + ".join(config_name) if config_name else "Sin features"
print(f"\n{'='*60}")
print(f"EJECUTANDO HRM MEJORADO - CONFIGURACIÓN: {config_str}")
print(f"{'='*60}")

# Ajustar hiperparámetros según el tipo de features
if USE_EMBEDDINGS and not USE_GEO:
    # Solo embeddings - modelo más grande para aprovechar la riqueza semántica
    hidden_high, hidden_low, plan_dim = 512, 1024, 128
    K, lr, batch = 20, 1e-4, 256
    print("Configuración optimizada para EMBEDDINGS")

elif USE_GEO and not USE_EMBEDDINGS:
    # Solo geo - modelo más pequeño pero profundo
    hidden_high, hidden_low, plan_dim = 256, 512, 64
    K, lr, batch = 16, 2e-4, 512
    print("Configuración optimizada para VARIABLES GEOESPACIALES")

elif USE_EMBEDDINGS and USE_GEO:
    # Ambos - configuración balanceada
    hidden_high, hidden_low, plan_dim = 384, 768, 96
    K, lr, batch = 18, 1.5e-4, 384
    print("Configuración optimizada para EMBEDDINGS + GEO")

else:
    # Solo coords u otra combinación - configuración por defecto
    hidden_high, hidden_low, plan_dim = 256, 512, 64
    K, lr, batch = 12, 2e-4, 512
    print("Configuración por defecto")

print(f"Parámetros: Hidden({hidden_high},{hidden_low}), Plan({plan_dim}), K={K}, LR={lr}, Batch={batch}")

# Entrenar HRM mejorado con configuración adaptiva
hrm_improved, history = train_improved_hrm(
    Xtr, ytr, Xva, yva,
    epochs=300, batch=batch, lr=lr,
    hidden_high=hidden_high, hidden_low=hidden_low, plan_dim=plan_dim,
    K=K, p_dropout=0.15,
    deep_supervision=True, use_halting=True,
    alpha_ds=0.6, beta_halt=0.01, patience=30
)

print("\nExtrayendo embeddings mejorados...")
# Extraer embeddings (pool ponderado por halting; concatena originales para "Wide & Deep")
E_tr = hrm_transform(hrm_improved, Xtr, batch=1024, pool_mode="weighted_last", concat_original=True)
E_va = hrm_transform(hrm_improved, Xva, batch=1024, pool_mode="weighted_last", concat_original=True)
E_te = hrm_transform(hrm_improved, Xte, batch=1024, pool_mode="weighted_last", concat_original=True)

print(f"Embeddings extraídos - Shape: {E_tr.shape}")

# Entrenar Random Forest mejorado
print("Entrenando Random Forest mejorado...")
rf_improved = RandomForestRegressor(
    n_estimators=1200,  # Más árboles
    max_depth=25,       # Profundidad controlada
    max_features="sqrt",
    min_samples_split=3,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True
)

rf_improved.fit(E_tr, ytr)

# Predicciones
pred_va = rf_improved.predict(E_va)
pred_te = rf_improved.predict(E_te)

# Resultados finales con configuración específica
print(f"\n{'='*60}")
print(f"RESULTADOS HRM MEJORADO - {config_str}")
print(f"{'='*60}")

r2_val = r2_score(yva, pred_va)
r2_test = r2_score(yte, pred_te)

print(f"Configuración: {config_str}")
print(f"Features utilizadas: {len(feats)}")
print(f"Modelo: H({hidden_high},{hidden_low}), P({plan_dim}), K={K}")
print(f"\nRESULTADOS:")
print(f"Valid R²: {r2_val:.4f}")
print(f"Test R²:  {r2_test:.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(yte, pred_te)):.6f}")
print(f"Test MAE:  {mean_absolute_error(yte, pred_te):.6f}")
print(f"RF OOB Score: {rf_improved.oob_score_:.4f}")

# Comparación con tu resultado original (que era solo Geo)
if USE_GEO and not USE_EMBEDDINGS:
    print(f"\nCOMPARACIÓN (vs tu resultado original Geo):")
    print(f"Original Test R²: 0.9169")
    print(f"Mejorado Test R²: {r2_test:.4f}")
    print(f"Incremento: +{(r2_test - 0.9169):.4f}")

# Meta alcanzada
if r2_test >= 0.97:
    print("¡OBJETIVO ALCANZADO! R² >= 0.97")
else:
    print(f"Progreso hacia R² 0.97: {(r2_test/0.97)*100:.1f}%")

# Guardar resultados en Drive (opcional)
results_summary = {
    'config': config_str,
    'features_count': len(feats),
    'use_embeddings': USE_EMBEDDINGS,
    'use_geo': USE_GEO,
    'use_coords': USE_COORDS,
    'r2_val': float(r2_val),
    'r2_test': float(r2_test),
    'rmse_test': float(np.sqrt(mean_squared_error(yte, pred_te))),
    'mae_test': float(mean_absolute_error(yte, pred_te)),
    'rf_oob': float(rf_improved.oob_score_),
    'model_params': {
        'hidden_high': hidden_high,
        'hidden_low': hidden_low,
        'plan_dim': plan_dim,
        'K': K,
        'lr': lr,
        'batch': batch
    }
}



EJECUTANDO HRM MEJORADO - CONFIGURACIÓN: Embeddings + Geo
Configuración optimizada para EMBEDDINGS + GEO
Parámetros: Hidden(384,768), Plan(96), K=18, LR=0.00015, Batch=384


C:\Users\cuent\anaconda3\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.15 and num_layers=1
  warnings.warn(


Iniciando entrenamiento HRM mejorado...
Epoch 0: Train Loss 0.085215, Val Loss 0.023805
Epoch 20: Train Loss 0.001565, Val Loss 0.000623
Epoch 40: Train Loss 0.000562, Val Loss 0.000310
Epoch 60: Train Loss 0.000283, Val Loss 0.000154
Epoch 80: Train Loss 0.000187, Val Loss 0.000113
Epoch 100: Train Loss 0.000133, Val Loss 0.000102
Epoch 120: Train Loss 0.000098, Val Loss 0.000088
Epoch 140: Train Loss 0.000083, Val Loss 0.000098
Epoch 160: Train Loss 0.000071, Val Loss 0.000083
Epoch 180: Train Loss 0.000061, Val Loss 0.000077
Epoch 200: Train Loss 0.000056, Val Loss 0.000082
Epoch 220: Train Loss 0.000048, Val Loss 0.000085
Epoch 240: Train Loss 0.000047, Val Loss 0.000075
Epoch 260: Train Loss 0.000040, Val Loss 0.000072
Epoch 280: Train Loss 0.000035, Val Loss 0.000071

Extrayendo embeddings mejorados...
Embeddings extraídos - Shape: (45041, 1318)
Entrenando Random Forest mejorado...

RESULTADOS HRM MEJORADO - Embeddings + Geo
Configuración: Embeddings + Geo
Features utilizadas: 70

NameError: name 'joblib' is not defined

Extrayendo embeddings mejorados...
Embeddings extraídos - Shape: (45041, 1318)
Entrenando Random Forest mejorado...

============================================================
RESULTADOS HRM MEJORADO - Embeddings + Geo
============================================================
Configuración: Embeddings + Geo
Features utilizadas: 70
Modelo: H(384,768), P(96), K=18

RESULTADOS:
Valid R²: 0.9635
Test R²:  0.9467
Test RMSE: 0.016848
Test MAE:  0.005166
RF OOB Score: 0.9946
Progreso hacia R² 0.97: 97.6%

In [9]:
import joblib 

# Guardar
model_path = 'HRM_rf.pkl'
joblib.dump(rf_improved, model_path)
print(f"\nModelo HRM + Random Forest guardado como '{model_path}'")

joblib.dump(rf_improved, os.path.join(parent_directory, 'models', 'HRM_rf.pkl'), compress=('lzma', 9))


# Guardar el resultado en un nuevo CSV
nombre_archivo_salida = "resultados_HRM.csv"
results_summary.to_csv(os.path.join(parent_directory,'data',nombre_archivo_salida), index=False) # index=False evita guardar el índice de pandas como una columna

print(f"\nDataFrame combinado guardado exitosamente como '{nombre_archivo_salida}' en '{os.path.join(parent_directory,'data',nombre_archivo_salida)}'")


Modelo HRM + Random Forest guardado como 'HRM_rf.pkl'


AttributeError: 'dict' object has no attribute 'to_csv'

In [11]:
# os.path.join() maneja correctamente las barras de ruta para diferentes SO (Windows, Linux, macOS)
file_path = os.path.join(parent_directory, 'data', 'df_massanassa.csv')


# Cargar el archivo CSV
df_massanassa = pd.read_csv(file_path)


# Verificar la estructura del DataFrame (equivalente a str(data) en R)
print(df_massanassa.info())

# os.path.join() maneja correctamente las barras de ruta para diferentes SO (Windows, Linux, macOS)
file_path = os.path.join(parent_directory, 'data', 'df_catarroja.csv')


# Cargar el archivo CSV
df_catarroja = pd.read_csv(file_path)


# Verificar la estructura del DataFrame (equivalente a str(data) en R)
print(df_catarroja.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17590 entries, 0 to 17589
Data columns (total 85 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   POINT_X  17590 non-null  float64
 1   POINT_Y  17590 non-null  float64
 2   PLA      17590 non-null  float64
 3   A        17590 non-null  float64
 4   CIB      17590 non-null  float64
 5   CICCB    17590 non-null  float64
 6   DBB      17590 non-null  float64
 7   DOB      17590 non-null  float64
 8   P        17590 non-null  float64
 9   CUS_1    0 non-null      float64
 10  CUS_2    0 non-null      float64
 11  CUS_3    0 non-null      float64
 12  CUS_4    0 non-null      float64
 13  CUS_5    0 non-null      float64
 14  OC_1     0 non-null      float64
 15  OC_2     0 non-null      float64
 16  OC_3     0 non-null      float64
 17  OC_4     0 non-null      float64
 18  lon      17590 non-null  float64
 19  lat      17590 non-null  float64
 20  emb_0    17590 non-null  float64
 21  emb_1    175

####################### P`ROBAR OTROS MUINICIPIOS


import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# Separar X e y
X_massanassa = df_massanassa[feats].values
y_massanassa = df_massanassa["PLA"].values

X_catarroja = df_catarroja[feats].values
y_catarroja = df_catarroja["PLA"].values

# Extraer embeddings con el HRM entrenado
E_massanassa = hrm_transform(hrm_improved, X_massanassa, batch=1024,
                             pool_mode="weighted_last", concat_original=True)
E_catarroja  = hrm_transform(hrm_improved, X_catarroja, batch=1024,
                             pool_mode="weighted_last", concat_original=True)

# Predicciones con el Random Forest entrenado
pred_massanassa = rf_improved.predict(E_massanassa)
pred_catarroja  = rf_improved.predict(E_catarroja)

# --- MÉTRICAS ---
def eval_metrics(y_true, y_pred, nombre):
    r2  = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"\nResultados en {nombre}:")
    print(f"R²:   {r2:.4f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")
    return {"R2": r2, "RMSE": rmse, "MAE": mae}

metrics_massanassa = eval_metrics(y_massanassa, pred_massanassa, "Massanassa")
metrics_catarroja  = eval_metrics(y_catarroja, pred_catarroja, "Catarroja")

In [13]:

metrics_massanassa = eval_metrics(y_massanassa, pred_massanassa, "Massanassa")
metrics_catarroja  = eval_metrics(y_catarroja, pred_catarroja, "Catarroja")


Resultados en Massanassa:
R²:   -4.3128
RMSE: 1.339250
MAE:  1.206374

Resultados en Catarroja:
R²:   -4.3128
RMSE: 1.339250
MAE:  1.206374



Resultados en Massanassa:
R²:   -4.3128
RMSE: 1.339250
MAE:  1.206374

Resultados en Catarroja:
R²:   -4.3128
RMSE: 1.339250
MAE:  1.206374